## Potencial Seno DF (Fortran), Numerov (Python)

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Intentar cargar los datos de Fortran
try:
    with open("data_seno", "r") as f:
        first_line = f.readline()
        energies_fortran = np.fromstring(first_line, sep=' ')

    data = np.loadtxt("data_seno", skiprows=1)
    x_val = data[:, 0]
    h = x_val[1] - x_val[0]
except Exception as e:
    print(f"Asegúrate de ejecutar el código Fortran primero. Error: {e}")

def solve_numerov_seno(energy, x_array):
    N_pts = len(x_array)
    dx = x_array[1] - x_array[0]
    psi = np.zeros(N_pts)

    # Ecuación: psi'' + 2(E - sin(x))psi = 0
    def k2(pos): return 2.0 * (energy - np.sin(pos))

    psi[0] = 0.0
    psi[1] = 1e-7
    f = (dx**2) / 12.0

    for i in range(1, N_pts - 1):
        k_curr, k_prev, k_next = k2(x_array[i]), k2(x_array[i-1]), k2(x_array[i+1])
        m_curr = 2.0 * (1.0 - 5.0 * f * k_curr) * psi[i]
        m_prev = (1.0 + f * k_prev) * psi[i-1]
        denom = (1.0 + f * k_next)
        psi[i+1] = (m_curr - m_prev) / denom
        if abs(psi[i+1]) > 1e2: psi[i+1] = np.sign(psi[i+1]) * 1e2

    psi_sq = psi**2
    return psi_sq / np.trapezoid(psi_sq, x_array)

def plot_final(n):
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # Datos de Fortran (Diferencias Finitas)
    e_fd = energies_fortran[n]
    prob_fd = data[:, n+1]

    # Datos de Numerov usando la energía de Fortran
    prob_num = solve_numerov_seno(e_fd, x_val)

    # Eje izquierdo: Probabilidad
    ax1.plot(x_val, prob_fd, 'b-', lw=2, label=f'Fortran (DF) n={n}')
    ax1.plot(x_val, prob_num, 'g--', lw=2, label=f'Python (Numerov) n={n}')
    ax1.fill_between(x_val, np.sin(x_val)*0.05, color='gray', alpha=0.1, label='V(x)=sin(x) (esc)')

    ax1.set_ylabel(r'$|\psi|^2$', fontsize=12, color='blue')
    ax1.set_xlabel('Posición (x)', fontsize=12)
    ax1.set_ylim(-0.05, 0.6)
    ax1.grid(True, alpha=0.2)

    # Eje derecho: Energía
    ax2 = ax1.twinx()
    ax2.set_ylim(min(energies_fortran)-0.5, max(energies_fortran)+0.5)
    ax2.axhline(e_fd, color='red', lw=1.5, alpha=0.6, label=f'E = {e_fd:.4f}')
    ax2.set_yticks([e_fd])
    ax2.set_yticklabels([f'E:{e_fd:.4f}'])
    ax2.set_ylabel('Energía', color='red')

    # Título y Leyenda
    plt.title(f'Comparativa: Fortran (diagotri) vs Numerov | Potencial $\\sin(x)$', fontsize=14)
    lines, labels = ax1.get_legend_handles_labels()
    ax1.legend(lines, labels, loc='upper right')

    plt.show()

n_slider = widgets.IntSlider(value=0, min=0, max=5, step=1, description='Nivel n:')
widgets.interactive(plot_final, n=n_slider)


interactive(children=(IntSlider(value=0, description='Nivel n:', max=5), Output()), _dom_classes=('widget-inte…

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ========================================================
# 1. CARGA DE DATOS DESDE FORTRAN
# ========================================================
try:
    with open("data_armonico", "r") as f:
        first_line = f.readline()
        energies_fortran = np.fromstring(first_line, sep=' ')

    data = np.loadtxt("data_armonico", skiprows=1)
    x_val = data[:, 0]
    h = x_val[1] - x_val[0]
    N = len(x_val)
except Exception as e:
    print(f"Error: Ejecuta el Fortran primero para generar 'data_armonico'. {e}")

# ========================================================
# 2. MÉTODO DE NUMEROV (Integración Inversa para Estabilidad)
# ========================================================
def solve_numerov_ho(energy, x_array, n):
    N_pts = len(x_array)
    dx = x_array[1] - x_array[0]
    psi = np.zeros(N_pts)
    f = (dx**2) / 12.0

    # k2 = 2 * (E - V) donde V = 0.5 * x^2
    k2 = 2.0 * (energy - 0.5 * x_array**2)

    # Integración desde el borde derecho hacia el centro (Estable)
    psi[-1] = 0.0
    psi[-2] = 1e-10

    half = N_pts // 2
    for i in range(N_pts - 2, 0, -1):
        term_curr = 2.0 * (1.0 - 5.0 * f * k2[i]) * psi[i]
        term_next = (1.0 + f * k2[i+1]) * psi[i+1]
        psi[i-1] = (term_curr - term_next) / (1.0 + f * k2[i-1])
        # Evitar overflow antes de normalizar
        if abs(psi[i-1]) > 1e15: psi[i-1] = np.sign(psi[i-1]) * 1e15

    # Aplicar paridad según el nivel n
    if n % 2 == 0: # Simétrico
        psi[:half] = psi[half:][::-1]
    else:          # Anti-simétrico
        psi[:half] = -psi[half:][::-1]

    psi_sq = psi**2
    return psi_sq / np.trapezoid(psi_sq, x_array)

# ========================================================
# 3. INTERFAZ GRÁFICA FINAL
# ========================================================
def plot_final(n):
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # Datos de Fortran (Diferencias Finitas)
    e_fd = energies_fortran[n]
    prob_fd = data[:, n+1]

    # Datos de Numerov usando la energía de Fortran
    prob_num = solve_numerov_ho(e_fd, x_val, n)

    # --- EJE IZQUIERDO: DENSIDAD DE PROBABILIDAD ---
    lns1 = ax1.plot(x_val, prob_fd, color='blue', lw=2, label=f'Fortran (DF) n={n}')
    lns2 = ax1.plot(x_val, prob_num, color='green', linestyle='--', lw=2, label=f'Python (Numerov) n={n}')

    # Sombreado del potencial V(x) = 0.5 * x^2 (Escalado para visualización)
    v_plot = 0.5 * x_val**2
    ax1.fill_between(x_val, v_plot * 0.02, color='gray', alpha=0.15, label=r'Potencial $V(x)=\frac{1}{2}x^2$ (esc)')

    ax1.set_ylabel(r'$|\psi|^2$', fontsize=12, color='blue')
    ax1.set_xlabel('Posición (x)', fontsize=12)
    ax1.set_ylim(-0.05, 0.6)
    ax1.set_xlim(-6, 6)
    ax1.grid(True, alpha=0.2)

    # --- EJE DERECHO: ENERGÍA ---
    ax2 = ax1.twinx()
    # Rango de energía dinámico según los datos
    ax2.set_ylim(0, 6)

    # Línea horizontal de energía (Basada en Fortran)
    lns3 = ax2.axhline(e_fd, color='red', lw=1.5, alpha=0.6, label=f'E = {e_fd:.4f}')

    # Marca de energía teórica (n + 0.5) para comparar visualmente
    e_teo = n + 0.5
    ax2.axhline(e_teo, color='darkred', lw=1, linestyle=':', alpha=0.3)

    ax2.set_yticks([e_fd, e_teo])
    ax2.set_yticklabels([f'E_FD:{e_fd:.4f}', f'E_Teo:{e_teo:.1f}'])
    ax2.set_ylabel('Energía', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')

    # Título y Leyenda combinada
    plt.title(f'Oscilador Armónico: Fortran (diagotri) vs Numerov (n={n})', fontsize=14)

    # Unificar todas las leyendas en ax1
    lines = lns1 + lns2 + [lns3]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.show()

n_slider = widgets.IntSlider(value=0, min=0, max=5, step=1, description='Nivel n:',
                             style={'description_width': 'initial'})
display(widgets.interactive(plot_final, n=n_slider))


interactive(children=(IntSlider(value=0, description='Nivel n:', max=5, style=SliderStyle(description_width='i…

test

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ========================================================
# 1. PARÁMETROS DEL POTENCIAL Y MALLA
# ========================================================
Rmin, Rmax = -5.0, 5.0
N = 1000
x = np.linspace(Rmin, Rmax, N)
h = x[1] - x[0]

# Parámetros del potencial: V(x) = a*x^(2*alpha) + b*x^2
a = 0.5
b = -2.0  # b < 0 crea el doble pozo
alpha = 2

V = a * x**(2*alpha) + b * x**2

# ========================================================
# 2. SOLUCIÓN DIFERENCIAS FINITAS (Referencia)
# ========================================================
def solve_fd():
    dx = x[1] - x[0]
    # Hamiltoniano: T = -0.5 * d2/dx2, V(x)
    main_diag = 1.0 / dx**2 + V
    off_diag = -0.5 / dx**2 * np.ones(N - 1)

    H = np.diag(main_diag) + np.diag(off_diag, -1) + np.diag(off_diag, 1)
    evals, evecs = np.linalg.eigh(H)

    densities = []
    for i in range(6):
        psi_sq = evecs[:, i]**2
        norm = np.trapezoid(psi_sq, x)
        densities.append(psi_sq / norm)

    return evals[:6], densities

evals_fd, densities_fd = solve_fd()

# ========================================================
# 3. MÉTODO DE NUMEROV (Integración Inversa)
# ========================================================
def solve_numerov(n, energy):
    psi = np.zeros(N)
    f = (h**2) / 12.0
    k2 = 2.0 * (energy - V)

    # Integración desde el borde hacia el centro
    psi[-1] = 0.0
    psi[-2] = 1e-10

    half = N // 2
    for i in range(N-2, 0, -1):
        term_curr = 2.0 * (1.0 - 5.0 * f * k2[i]) * psi[i]
        term_next = (1.0 + f * k2[i+1]) * psi[i+1]
        psi[i-1] = (term_curr - term_next) / (1.0 + f * k2[i-1])
        if abs(psi[i-1]) > 1e15: psi[i-1] = np.sign(psi[i-1]) * 1e15

    # Paridad según el estado n
    if n % 2 == 0:
        psi[:half] = psi[half:][::-1]
    else:
        psi[:half] = -psi[half:][::-1]

    prob = psi**2
    return prob / np.trapezoid(prob, x)

# ========================================================
# 4. INTERFAZ INTERACTIVA
# ========================================================
def update_plot(n):
    fig, ax1 = plt.subplots(figsize=(12, 7))

    e_fd = evals_fd[n]
    y_fd = densities_fd[n]
    y_num = solve_numerov(n, e_fd)

    # Probabilidad
    lns1 = ax1.plot(x, y_fd, color='blue', lw=3, label=f'DF: $|\\psi|^2$ (n={n})', alpha=0.5)
    lns2 = ax1.plot(x, y_num, color='green', linestyle='--', lw=2, label=f'Numerov: $|\\psi|^2$ (n={n})')

    # Dibujar el potencial de fondo
    ax1.fill_between(x, V * 0.05, color='gray', alpha=0.1, label='V(x) = 0.5x^4 - 2x^2 (esc)')

    ax1.set_ylim(-0.02, 0.8)
    ax1.set_xlim(-4, 4)
    ax1.set_xlabel('Posición (x)')
    ax1.set_ylabel(r'Probabilidad $|\psi|^2$')
    ax1.grid(True, alpha=0.2)

    # Energía
    ax2 = ax1.twinx()
    ax2.set_ylim(min(evals_fd)-1, max(evals_fd)+1)
    lns3 = ax2.axhline(e_fd, color='red', lw=1.5, alpha=0.6, label=f'E = {e_fd:.4f}')
    ax2.set_yticks([e_fd])
    ax2.set_ylabel('Energía', color='red')

    plt.title(f'Potencial de Doble Pozo: Diferencias Finitas vs Numerov (n={n})', fontsize=14)
    lines = lns1 + lns2 + [lns3]
    ax1.legend(lines, [l.get_label() for l in lines], loc='upper right')
    plt.show()

n_slider = widgets.IntSlider(value=0, min=0, max=5, step=1, description='Nivel n:')
display(widgets.interactive(update_plot, n=n_slider))


interactive(children=(IntSlider(value=0, description='Nivel n:', max=5), Output()), _dom_classes=('widget-inte…